# 02 — Modelado: U-Net para agua de inundación (versión final)

Este notebook contiene **solo el camino final**, listo para "Ejecutar todo" en una sesión nueva de Colab:
1. Cargar los chips, normalizar con train y armar el Dataset.
2. U-Net propia (con opción de *attention gates*), loss **BCE+Dice** con máscara y métricas de la clase agua.
3. **E3** (U-Net + BCE+Dice, solo chips dorados) = mejor modelo de la ablación.
4. **E6** = la misma receta con **Attention U-Net**. Se queda solo si mejora a E3 en valid.
5. **E7** = la arquitectura ganadora con **100 épocas** en vez de 40. Se queda solo si mejora en valid.
6. **Modelo final**: la receta ganadora, reentrenada con **train + valid** (336 chips).
7. Evaluación final en **test** y **Bolivia**.

> Los experimentos E1–E5 (otras losses y etiquetas débiles) están en `02b_ablaciones.ipynb`. Sus resultados ya están en Drive y aquí solo se leen.
>
> **Si un experimento ya está en Drive, no se vuelve a entrenar** (se lee su `history.csv`). Si la sesión se apaga, vuelve a "Ejecutar todo": solo entrena lo que falta.

### Decisiones que salen de la exploración (`inventario_chips.csv`)

| Hallazgo | Decisión |
|---|---|
| 6 chips de Ghana tienen ≥ 99 % de píxeles −1, y el chip `Paraguay_34417` tiene la imagen 100 % NaN | Se **excluyen** (7 chips). Quedan 250 / 86 / 88 / 15 |
| Hay máscaras con etiqueta donde la imagen es NaN (sin medición de radar) | Esos píxeles pasan a **−1**: no se puede predecir sin imagen |
| Solo ≈ 10.6 % de los píxeles es agua; mediana 2.5 % por chip; 52 chips sin agua | Métricas de la **clase agua** (IoU, F1), no accuracy |
| Diferencia agua–tierra en VV: mediana 5.2 dB; en 99 chips < 3 dB | Un umbral fijo no alcanza → justifica la red (y la comparamos con Otsu) |

## 0. Preparación

In [ ]:
!pip install -q rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = '/content/drive/MyDrive/tarea2_floods'
DATA = f'{ROOT}/data/sen1floods11'
OUT  = f'{ROOT}/outputs'
os.makedirs(OUT, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '⚠️ activa la GPU: Entorno de ejecución → Cambiar tipo → T4')

La línea base necesita las máscaras del umbral Otsu (≈ 3 MB). Se descargan una sola vez.

In [ ]:
if not os.path.exists(f'{DATA}/S1OtsuLabelHand'):
    !gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1OtsuLabelHand "{DATA}/"
print('S1OtsuLabelHand:', len(os.listdir(f'{DATA}/S1OtsuLabelHand')), 'archivos (esperado 446)')

## 1. Lista de chips y exclusiones

Unimos las particiones con el inventario de la exploración y marcamos los chips inservibles.

In [ ]:
inv = pd.read_csv(f'{DATA}/inventario_chips.csv')
inv['chip'] = inv['evento'] + '_' + inv['chip_id'].astype(str)

parts = []
for s in ['train', 'valid', 'test', 'bolivia']:
    df = pd.read_csv(f'{DATA}/splits/flood_{s}_data.csv', header=None, names=['s1_file', 'label_file'])
    df['split'] = s
    parts.append(df)
chips = pd.concat(parts, ignore_index=True)
chips['chip'] = chips['s1_file'].str.replace('_S1Hand.tif', '', regex=False)
chips = chips.merge(inv[['chip', 'evento', 'pct_agua', 'pct_sin_dato', 'pct_nan']], on='chip', how='left')

chips['excluir'] = (chips['pct_sin_dato'] >= 99) | (chips['pct_nan'] >= 99)
chips[chips['excluir']]

In [ ]:
chips = chips[~chips['excluir']].reset_index(drop=True)
chips['split'].value_counts()      # esperado: train 250, test 88, valid 86, bolivia 15

## 2. Cargar los chips en memoria

Leer 400+ archivos desde Drive en cada época sería lento. Los leemos **una vez** y los guardamos en un diccionario (`CACHE`): ≈ 1 GB de RAM.

`read_chip` devuelve la imagen cruda en dB (con NaN) y la máscara con −1 también donde la imagen no tiene dato.

In [ ]:
def read_chip(img_path, label_path):
    """Imagen cruda (2, 512, 512) en dB con NaN, y máscara int8 con -1 / 0 / 1."""
    with rasterio.open(img_path) as src:
        img = src.read().astype('float32')
    with rasterio.open(label_path) as src:
        lab = src.read(1).astype('int8')
    sin_imagen = np.isnan(img).any(axis=0)
    lab[sin_imagen] = -1          # sin medición de radar → ese píxel no se evalúa
    return img, lab


CACHE = {}
for r in tqdm(chips.itertuples(), total=len(chips)):
    CACHE[r.chip] = read_chip(f'{DATA}/S1Hand/{r.s1_file}', f'{DATA}/LabelHand/{r.label_file}')
print('Chips en memoria:', len(CACHE))

## 3. Normalización (solo con train)

La red aprende mejor con valores alrededor de 0. Pasos:
1. **Recortar extremos** (percentiles 0.1 % y 99.9 %): unos pocos píxeles muy brillantes (ciudades) o muy oscuros no deben dominar.
2. **Estandarizar**: `(x − media) / desviación` por banda.
3. Los NaN pasan a 0 (y en la máscara ya son −1, así que no cuentan).

Las estadísticas salen **solo de train** (una muestra de 20,000 píxeles por chip). Usar valid/test aquí sería "espiar" las respuestas.

In [ ]:
rng = np.random.default_rng(SEED)
muestras = []
for chip in chips.loc[chips['split'] == 'train', 'chip']:
    img, _ = CACHE[chip]
    px = img.reshape(2, -1)
    px = px[:, ~np.isnan(px).any(axis=0)]
    idx = rng.choice(px.shape[1], size=min(20000, px.shape[1]), replace=False)
    muestras.append(px[:, idx])
muestras = np.concatenate(muestras, axis=1)

LO = np.percentile(muestras, 0.1, axis=1)
HI = np.percentile(muestras, 99.9, axis=1)
recortado = np.clip(muestras, LO[:, None], HI[:, None])
STATS = {'lo': LO.tolist(), 'hi': HI.tolist(),
         'mean': recortado.mean(axis=1).tolist(), 'std': recortado.std(axis=1).tolist()}
json.dump(STATS, open(f'{OUT}/norm_stats.json', 'w'), indent=2)    # se reutiliza para Lima y Bolivia

pd.DataFrame(STATS, index=['VV', 'VH']).round(2)

In [ ]:
def normalize(img):
    """dB crudo → recorta extremos, estandariza por banda y pone 0 donde no hay dato."""
    lo, hi = np.array(STATS['lo'])[:, None, None], np.array(STATS['hi'])[:, None, None]
    mean, std = np.array(STATS['mean'])[:, None, None], np.array(STATS['std'])[:, None, None]
    x = (np.clip(img, lo, hi) - mean) / std
    return np.nan_to_num(x, nan=0.0).astype('float32')

## 4. Dataset y aumentaciones

- **Train:** espejos y rotaciones de 90° al azar (el agua no tiene "arriba" ni "abajo") + **recorte aleatorio de 256×256**. El recorte da más variedad por época y entrena ≈ 4 veces más rápido.
- **Valid / test:** chip completo de 512×512, sin aumentaciones (la U-Net acepta cualquier tamaño múltiplo de 16).
- Sin cambios de brillo ni color: en radar el brillo **es** la información.

In [ ]:
def augment(img, lab, rng):
    if rng.random() < 0.5:
        img, lab = img[:, :, ::-1], lab[:, ::-1]                     # espejo horizontal
    if rng.random() < 0.5:
        img, lab = img[:, ::-1, :], lab[::-1, :]                     # espejo vertical
    k = int(rng.integers(4))
    return np.rot90(img, k, axes=(1, 2)), np.rot90(lab, k)          # rotación 0/90/180/270°


def random_crop(img, lab, size, rng):
    H, W = lab.shape
    y, x = int(rng.integers(0, H - size + 1)), int(rng.integers(0, W - size + 1))
    return img[:, y:y + size, x:x + size], lab[y:y + size, x:x + size]


class FloodDataset(Dataset):
    def __init__(self, df, train=False, crop=None):
        self.chips = df['chip'].tolist() if isinstance(df, pd.DataFrame) else list(df)    # DataFrame o lista de nombres
        self.train = train
        self.crop = crop

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, i):
        img, lab = CACHE[self.chips[i]]
        img = normalize(img)
        if self.train:
            rng = np.random.default_rng(int(torch.randint(2**31, (1,))))    # semilla desde torch → reproducible
            img, lab = augment(img, lab, rng)
            if self.crop:
                img, lab = random_crop(img, lab, self.crop, rng)
        img = torch.from_numpy(np.ascontiguousarray(img))
        lab = torch.from_numpy(np.ascontiguousarray(lab).astype('int64'))
        return img, lab

In [ ]:
def show_mask(ax, lab, title):
    rgb = np.zeros((*lab.shape, 3))
    rgb[lab == 0] = [0.89, 0.84, 0.72]      # no agua (beige)
    rgb[lab == 1] = [0.12, 0.44, 0.70]      # agua (azul)
    rgb[lab == -1] = [0.60, 0.60, 0.60]     # sin dato (gris)
    ax.imshow(rgb); ax.set_title(title); ax.axis('off')

## 5. U-Net (implementación propia) con *attention gates* opcionales

- **Entrada: 2 canales** (VV, VH). **Salida: 1 canal** (logit de "agua"); `logit > 0` ⇔ probabilidad > 0.5.
- Filtros `(32, 64, 128, 256)` → ≈ 7.8 M parámetros. Con 250 chips, una red más grande memoriza más fácil.

| Paso | Forma con entrada 256×256 |
|---|---|
| Entrada | (B, 2, 256, 256) |
| Bajada 1–4 (+ guarda skip) | 32 @ 256 → 64 @ 128 → 128 @ 64 → 256 @ 32 |
| Fondo (bottleneck) | 512 @ 16 |
| Subida 4–1 (+ concatena skip) | 256 @ 32 → 128 @ 64 → 64 @ 128 → 32 @ 256 |
| Cabeza 1×1 | (B, 1, 256, 256) |

### ¿Qué es una *attention gate*? (Oktay et al., 2018, *Attention U-Net*)

En la U-Net normal, cada *skip* pasa **todo** lo que vio el encoder al decoder: bordes de ríos, pero también texturas de campos lisos y sombras que "parecen agua". La *attention gate* pone un **filtro aprendido** en cada skip:

1. Mira el skip (detalle fino, `x`) y lo que el decoder ya entendió de la escena (`g`, contexto grueso).
2. Calcula para cada píxel un peso entre 0 y 1: "¿este detalle es relevante para encontrar agua?".
3. Multiplica el skip por ese peso: lo irrelevante se apaga antes de concatenarse.

**Por qué la probamos aquí:** los dos fallos que vimos en E3 son (a) **agua inventada** en campos lisos (`Paraguay_76868`) y (b) **agua difusa o fina no vista** (`Paraguay_305760`). Las *gates* atacan justo eso: usar contexto para descartar zonas oscuras que no son agua y concentrarse en estructuras pequeñas. Cuesta solo ≈ 1 % más de parámetros.

**¿Es la arquitectura más óptima?** Nadie lo sabe antes de probar: con 250 chips, una red más compleja también puede rendir igual o peor. Por eso la regla es la misma de siempre: **decide valid**. Solo cambia la arquitectura (misma loss, datos y épocas que E3).

In [ ]:
def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class AttentionGate(nn.Module):
    """Pesa cada píxel del skip (x) con un valor 0–1 calculado a partir del decoder (g)."""
    def __init__(self, ch, inter):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.w_x = nn.Sequential(nn.Conv2d(ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, x, g):
        alpha = self.psi(F.relu(self.w_g(g) + self.w_x(x)))      # (B, 1, H, W): 0 = apagar, 1 = dejar pasar
        return x * alpha


class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, features=(32, 64, 128, 256), attention=False):
        super().__init__()
        self.down = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.down.append(double_conv(ch, f))
            ch = f
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = double_conv(ch, ch * 2)
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for f in reversed(features):
            self.up.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.dec.append(double_conv(f * 2, f))
        self.gates = nn.ModuleList([AttentionGate(f, f // 2) for f in reversed(features)]) if attention else None
        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for i, (up, dec, skip) in enumerate(zip(self.up, self.dec, reversed(skips))):
            x = up(x)
            if self.gates is not None:
                skip = self.gates[i](skip, x)      # filtra el skip con el contexto del decoder
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return self.head(x)      # logits (B, 1, H, W)

In [ ]:
for att in [False, True]:
    m = UNet(attention=att).to(DEVICE)
    salida = m(torch.randn(2, 2, 256, 256, device=DEVICE))
    print(f"{'Attention U-Net' if att else 'U-Net          '} → salida {tuple(salida.shape)} · {sum(p.numel() for p in m.parameters()):,} parámetros")

## 6. Funciones de pérdida con máscara

Todas **ignoran los píxeles −1**: se multiplican por `valid` (1 si el píxel cuenta, 0 si no). La ablación (E1–E3) eligió **`bce_dice`** = 0.5 · BCE + 0.5 · Dice: la Dice no se deja dominar por el 90 % de "no agua" y sube el recall; la BCE la estabiliza.

In [ ]:
def masked_bce(logits, target):
    valid = (target != -1).float()
    t = target.clamp(min=0).float()
    loss = F.binary_cross_entropy_with_logits(logits.squeeze(1), t, reduction='none')
    return (loss * valid).sum() / valid.sum().clamp(min=1)


def masked_dice(logits, target, eps=1.0):
    valid = (target != -1).float()
    p = torch.sigmoid(logits.squeeze(1)) * valid
    t = target.clamp(min=0).float() * valid
    inter = (p * t).sum()
    return 1 - (2 * inter + eps) / (p.sum() + t.sum() + eps)


LOSSES = {
    'bce': masked_bce,
    'dice': masked_dice,
    'bce_dice': lambda logits, target: 0.5 * masked_bce(logits, target) + 0.5 * masked_dice(logits, target),
}

## 7. Métricas de la clase agua

Se acumula la matriz de confusión sobre **todos los píxeles válidos** de un split y al final se calculan:

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| IoU | TP / (TP + FP + FN) | ¿Cuánto se solapan la mancha predicha y la real? (métrica principal) |
| Precisión | TP / (TP + FP) | De lo que marqué como agua, ¿cuánto era agua? |
| Recall | TP / (TP + FN) | Del agua real, ¿cuánta encontré? |
| F1 | 2·P·R / (P + R) | Balance entre precisión y recall |
| Omisión | FN / (TP + FN) | Agua real que no detecté (= 1 − recall) |
| Comisión | FP / (TP + FP) | Falsa agua (= 1 − precisión) |

In [ ]:
class WaterMetrics:
    """Matriz de confusión de la clase agua sobre los píxeles válidos (target != -1)."""
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        """pred: array bool (agua sí/no); target: array con -1 / 0 / 1 del mismo tamaño."""
        valid, agua = target != -1, target == 1
        self.tp += int(( pred &  agua & valid).sum())
        self.fp += int(( pred & ~agua & valid).sum())
        self.fn += int((~pred &  agua & valid).sum())
        self.tn += int((~pred & ~agua & valid).sum())

    def compute(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        return {
            'iou': tp / max(tp + fp + fn, 1),
            'f1': 2 * precision * recall / max(precision + recall, 1e-9),
            'precision': precision,
            'recall': recall,
            'omision': fn / max(tp + fn, 1),
            'comision': fp / max(tp + fp, 1),
            'accuracy': (tp + tn) / max(tp + fp + fn + tn, 1),
        }

    def confusion(self):
        return pd.DataFrame([[self.tn, self.fp], [self.fn, self.tp]],
                            index=['real: no agua', 'real: agua'], columns=['pred: no agua', 'pred: agua'])

## 8. Línea base: umbral de Otsu

`S1OtsuLabelHand` es lo que da el método clásico sobre los mismos chips. Es la vara mínima: **la U-Net debe superarla**.

In [ ]:
def eval_otsu(split):
    m = WaterMetrics()
    for chip in chips.loc[chips['split'] == split, 'chip']:
        with rasterio.open(f'{DATA}/S1OtsuLabelHand/{chip}_S1OtsuLabelHand.tif') as src:
            otsu = src.read(1)
        _, lab = CACHE[chip]
        m.update(otsu == 1, lab)
    return m


otsu_valid = eval_otsu('valid')
pd.Series(otsu_valid.compute()).round(4).to_frame('Otsu · valid')

## 9. Entrenamiento

- `train_one_epoch`: una pasada por train con *mixed precision* (más rápido en la T4).
- `evaluate`: loss + métricas en un split, sin aprender.
- `run_experiment`: entrena, evalúa en valid cada época, guarda el **mejor checkpoint por IoU** y el historial en `outputs/<nombre>/`.
- `load_model`: recarga el mejor checkpoint (sabe si era U-Net o Attention U-Net).

**Si el experimento ya está en Drive, no se vuelve a entrenar.** Para reentrenar uno a propósito: `force=True`.

**Si la sesión se apaga a mitad de un entrenamiento**, no se pierde: al final de cada época se guarda `last.pt` (pesos, optimizador, *learning rate* y el historial). Al volver a "Ejecutar todo", el experimento **sigue desde la época donde quedó**. Esto importa porque E7 dura 100 épocas.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, scaler):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    m = WaterMetrics()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
        total += loss_fn(logits.float(), y).item() * x.size(0)
        n += x.size(0)
        m.update((logits[:, 0] > 0).cpu().numpy(), y.cpu().numpy())
    return total / n, m

In [ ]:
valid_loader = DataLoader(FloodDataset(chips[chips['split'] == 'valid']), batch_size=4, shuffle=False, num_workers=2)


def run_experiment(name, loss_name='bce_dice', epochs=40, lr=1e-3, batch_size=8, crop=256, attention=False,
                   train_splits=('train',), seed=SEED, force=False):
    """Entrena y guarda en outputs/<name>/.
    - Si ya terminó (history.csv), no reentrena (force=True para reentrenar).
    - Si la sesión se cortó a mitad, retoma desde la última época guardada (last.pt).
    - Si 'valid' está en train_splits, ya no hay con qué elegir: se guarda el modelo de la última época."""
    exp_dir = f'{OUT}/{name}'
    if not force and os.path.exists(f'{exp_dir}/history.csv'):
        print(f'{name}: ya entrenado → se lee {exp_dir}/history.csv  (force=True para reentrenar)')
        return pd.read_csv(f'{exp_dir}/history.csv')
    os.makedirs(exp_dir, exist_ok=True)
    torch.manual_seed(seed)
    elige_con_valid = 'valid' not in train_splits

    train_loader = DataLoader(FloodDataset(chips[chips['split'].isin(train_splits)], train=True, crop=crop),
                              batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    model = UNet(attention=attention).to(DEVICE)
    loss_fn = LOSSES[loss_name]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    config = {'loss': loss_name, 'lr': lr, 'epochs': epochs, 'crop': crop, 'attention': attention,
              'train_splits': list(train_splits), 'seed': seed}

    history, best_iou, inicio = [], -1.0, 1
    if not force and os.path.exists(f'{exp_dir}/last.pt'):
        ck = torch.load(f'{exp_dir}/last.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
        torch.set_rng_state(ck['rng'])
        history, best_iou, inicio = ck['history'], ck['best_iou'], ck['epoch'] + 1
        print(f'{name}: se retoma desde la época {inicio} (la sesión anterior se cortó)')

    for epoch in range(inicio, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
        fila = {'epoch': epoch, 'train_loss': train_loss}
        if elige_con_valid:
            val_loss, val_m = evaluate(model, valid_loader, loss_fn)
            fila.update({'val_loss': val_loss, **val_m.compute()})
            mejora = fila['iou'] > best_iou
        else:
            mejora = epoch == epochs        # sin valid para elegir: se guarda la última época
        scheduler.step()
        history.append(fila)
        if mejora:
            best_iou = fila.get('iou', best_iou)
            torch.save({'epoch': epoch, 'model': model.state_dict(), 'config': config}, f'{exp_dir}/best.pt')
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'rng': torch.get_rng_state(),
                    'history': history, 'best_iou': best_iou}, f'{exp_dir}/last.pt')
        info = f"val {fila['val_loss']:.4f} | IoU {fila['iou']:.4f} | F1 {fila['f1']:.4f}" if elige_con_valid else 'sin valid'
        print(f"{epoch:3d} | train {train_loss:.4f} | {info} | {time.time() - t0:.0f}s {'✓ guardado' if mejora else ''}")

    history = pd.DataFrame(history)
    history.to_csv(f'{exp_dir}/history.csv', index=False)
    os.remove(f'{exp_dir}/last.pt')         # ya terminó: no hace falta el punto de retome
    print(f'\nTerminado → {exp_dir}/best.pt' + (f' (mejor IoU en valid: {best_iou:.4f})' if elige_con_valid else ' (última época)'))
    return history


def load_model(name):
    ckpt = torch.load(f'{OUT}/{name}/best.pt', map_location=DEVICE)
    model = UNet(attention=ckpt['config'].get('attention', False)).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    print(f"{name}: mejor época {ckpt['epoch']} · {ckpt['config']}")
    return model.eval()


@torch.no_grad()
def predict(model, chip):
    img, _ = CACHE[chip]
    x = torch.from_numpy(normalize(img))[None].to(DEVICE)
    return (model(x)[0, 0] > 0).cpu().numpy()

## 10. Lo que ya decidió la ablación (E1–E5, en `02b_ablaciones.ipynb`)

| | IoU valid | Recall | Comisión | Conclusión |
|---|---|---|---|---|
| Otsu | 0.507 | 0.716 | 0.366 | Línea base clásica |
| E1 · BCE | 0.623 | 0.700 | 0.149 | Conservadora e inestable |
| E2 · Dice | 0.640 | 0.756 | 0.194 | Sube el recall |
| **E3 · BCE+Dice** | **0.643** | **0.778** | 0.213 | **Mejor loss** |
| E4 · dorados + débiles | 0.577 | 0.690 | 0.221 | Las débiles empeoran |
| E5 · pre débiles → dorados | 0.620 | 0.741 | 0.208 | No supera a E3 |

La tabla completa se lee desde Drive (la guardó la ablación):

In [ ]:
pd.read_csv(f'{OUT}/resultados_valid.csv', index_col=0).round(4)

## 11. E3 — U-Net + BCE+Dice (modelo de referencia)

Ya está entrenado en Drive → esta celda solo lee su historial.

In [ ]:
hist_e3 = run_experiment('E3_bce_dice', loss_name='bce_dice')

## 12. E6 — Attention U-Net + BCE+Dice

Misma loss, mismos datos, mismas 40 épocas que E3. **Solo cambia la arquitectura.** Tarda un poco más que E3 (≈ 10 % más por época).

In [ ]:
hist_e6 = run_experiment('E6_attention', loss_name='bce_dice', attention=True)

## 13. E3 frente a E6 (valid): ¿qué arquitectura?

**Regla (la misma de toda la ablación):** E6 reemplaza a E3 solo si lo supera por **más de 0.02 de IoU** (la variación entre corridas es ≈ 0.01). Si quedan cerca, se queda **E3**: es más simple y ya está probado.

In [ ]:
EXPS = {'E3_bce_dice': 'E3 · U-Net', 'E6_attention': 'E6 · Attention U-Net'}
HIST = {name: pd.read_csv(f'{OUT}/{name}/history.csv') for name in EXPS}

filas, extra = {'Otsu': otsu_valid.compute()}, {}
for name, label in EXPS.items():
    _, met = evaluate(load_model(name), valid_loader, masked_bce)
    filas[label] = met.compute()
    h = HIST[name]
    extra[label] = {'mejor_epoca': int(h.loc[h['iou'].idxmax(), 'epoch']), 'iou_ult5': h['iou'].tail(5).mean()}

comparacion = pd.DataFrame(filas).T.join(pd.DataFrame(extra).T)
comparacion.to_csv(f'{OUT}/resultados_valid_E3_E6.csv')

diferencia = comparacion.loc['E6 · Attention U-Net', 'iou'] - comparacion.loc['E3 · U-Net', 'iou']
ATT = bool(diferencia > 0.02)                       # True → Attention U-Net
BASE = 'E6_attention' if ATT else 'E3_bce_dice'
print(f'E6 − E3 = {diferencia:+.4f} de IoU → arquitectura: {"Attention U-Net" if ATT else "U-Net"} ({BASE})')
comparacion.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, label in EXPS.items():
    ax1.plot(HIST[name]['epoch'], HIST[name]['iou'], label=label)
    ax2.plot(HIST[name]['epoch'], HIST[name]['recall'], label=label)
for ax, t in [(ax1, 'IoU en valid'), (ax2, 'Recall en valid')]:
    ax.axhline(otsu_valid.compute()[t.split()[0].lower()], color='gray', ls='--', label='Otsu')
    ax.set_title(t); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Los mismos 4 chips de valid de siempre

`Paraguay_76868` (campos lisos) y `Paraguay_305760` (agua difusa) son los casos donde las *attention gates* deberían notarse.

In [ ]:
ejemplos = chips[chips['split'] == 'valid'].sample(4, random_state=1)['chip']
modelos = {label: load_model(name) for name, label in EXPS.items()}

fig, axes = plt.subplots(4, 2 + len(modelos), figsize=(4 * (2 + len(modelos)), 16))
for row, chip in zip(axes, ejemplos):
    img, lab = CACHE[chip]
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0); row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    show_mask(row[1], lab, 'Real (LabelHand)')
    for ax, (label, m) in zip(row[2:], modelos.items()):
        show_mask(ax, np.where(lab == -1, -1, predict(m, chip)), label)
plt.tight_layout(); plt.show()

## 14. E7 — entrenar más tiempo (100 épocas)

Con el recorte de 256×256, **cada época solo ve ¼ de cada chip**: 40 épocas equivalen a ≈ 10 pasadas completas por los datos. Además, la loss de train y la de valid quedaron casi iguales (no hay sobreajuste) y el mejor checkpoint salía cerca del final. Todo indica que la red **podría aprender más si entrena más**.

E7 = la arquitectura ganadora de la sección 13, con la misma receta, pero **100 épocas** en vez de 40. El *learning rate* coseno se estira a las 100 épocas. Tarda ≈ 2.5 veces lo que E3.

In [ ]:
E7 = f"E7_{'attention' if ATT else 'unet'}_100ep"
hist_e7 = run_experiment(E7, loss_name='bce_dice', epochs=100, attention=ATT)

## 15. ¿Ayudaron las 100 épocas? (valid)

Misma regla: E7 se queda solo si supera a la versión de 40 épocas por **más de 0.02 de IoU**. Si no, entrenar más no vale el costo.

In [ ]:
EXPS2 = {BASE: f'{BASE} · 40 épocas', E7: f'{E7} · 100 épocas'}
HIST2 = {name: pd.read_csv(f'{OUT}/{name}/history.csv') for name in EXPS2}

filas, extra = {'Otsu': otsu_valid.compute()}, {}
for name, label in EXPS2.items():
    _, met = evaluate(load_model(name), valid_loader, masked_bce)
    filas[label] = met.compute()
    h = HIST2[name]
    extra[label] = {'mejor_epoca': int(h.loc[h['iou'].idxmax(), 'epoch']), 'iou_ult5': h['iou'].tail(5).mean()}

comparacion2 = pd.DataFrame(filas).T.join(pd.DataFrame(extra).T)
comparacion2.to_csv(f'{OUT}/resultados_valid_epocas.csv')

diferencia = comparacion2.loc[EXPS2[E7], 'iou'] - comparacion2.loc[EXPS2[BASE], 'iou']
ELEGIDO = E7 if diferencia > 0.02 else BASE
EPOCAS = 100 if ELEGIDO == E7 else 40
print(f'100 ép. − 40 ép. = {diferencia:+.4f} de IoU → modelo elegido en valid: {ELEGIDO} ({EPOCAS} épocas)')
comparacion2.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, label in EXPS2.items():
    ax1.plot(HIST2[name]['epoch'], HIST2[name]['iou'], label=label)
    ax2.plot(HIST2[name]['epoch'], HIST2[name]['recall'], label=label)
for ax, t in [(ax1, 'IoU en valid'), (ax2, 'Recall en valid')]:
    ax.axhline(otsu_valid.compute()[t.split()[0].lower()], color='gray', ls='--', label='Otsu')
    ax.set_title(t); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 16. Modelo final: reentrenar con train + valid

Ya está **todo decidido con valid**: loss (BCE+Dice), arquitectura (sección 13) y número de épocas (sección 15). Ahora valid ya no hace falta para elegir, así que se **suma al entrenamiento**: 250 + 86 = **336 chips dorados** (+34 % de datos).

- Misma receta exacta que `ELEGIDO`, con un **número fijo de épocas** (`EPOCAS`), y se guarda **la última época**: no hay con qué elegir otra.
- Este modelo **no tiene número de valid** (valid ahora es dato de entrenamiento). Su calidad se mide en test y Bolivia.
- Test y Bolivia **siguen sin tocarse**.

In [ ]:
FINAL = f'FINAL_{ELEGIDO}_train_valid'
hist_final = run_experiment(FINAL, loss_name='bce_dice', epochs=EPOCAS, attention=ATT, train_splits=('train', 'valid'))

# Fase 5: evaluación final (test y Bolivia)

Reglas de esta fase:
- El **test** y **Bolivia** se miran **una sola vez**, con los modelos ya decididos. No se reentrena ni se elige nada según estos números: si se hiciera, el test dejaría de ser una medida honesta.
- **Test** = mismos eventos que train (otros chips) → mide cómo rinde en lugares "conocidos" (**punto 3 del examen**).
- **Bolivia** = un evento que la red **nunca vio** → mide la **generalización** a un lugar nuevo, como será Lima.
- El modelo que se usa para Lima es **`FINAL`** (train + valid), decidido de antemano. `ELEGIDO` (solo train) aparece al lado solo para ver **cuánto aportaron los 86 chips extra**.

## 17. Métricas en test y Bolivia

In [ ]:
model_elegido = load_model(ELEGIDO)
model_final = load_model(FINAL)

def eval_split(model, split):
    loader = DataLoader(FloodDataset(chips[chips['split'] == split]), batch_size=4)
    return evaluate(model, loader, masked_bce)[1]

MET = {}
for split in ['test', 'bolivia']:
    MET[f'Otsu · {split}'] = eval_otsu(split)
    MET[f'Solo train · {split}'] = eval_split(model_elegido, split)
    MET[f'Red · {split}'] = eval_split(model_final, split)       # FINAL: train + valid

final = pd.DataFrame({k: m.compute() for k, m in MET.items()}).T
final.to_csv(f'{OUT}/resultados_final.csv')
final.round(4)

### Matrices de confusión (en píxeles)

Filas = lo real, columnas = lo que dice el modelo. `real: agua / pred: no agua` es la **omisión** (agua que no vio); `real: no agua / pred: agua` es la **comisión** (agua inventada).

In [ ]:
MET['Red · test'].confusion()

In [ ]:
MET['Red · bolivia'].confusion()

## 18. ¿En qué eventos falla más? (por evento)

Un solo número para todo el test esconde diferencias: puede ir muy bien en un país y mal en otro. Aquí se calcula el IoU de cada evento por separado (test + Bolivia), junto con el % de agua y de "sin dato" de sus chips.

In [ ]:
def otsu_mask(chip):
    with rasterio.open(f'{DATA}/S1OtsuLabelHand/{chip}_S1OtsuLabelHand.tif') as src:
        return src.read(1) == 1


filas = {}
for evento, grupo in chips[chips['split'].isin(['test', 'bolivia'])].groupby('evento'):
    m_otsu, m_red = WaterMetrics(), WaterMetrics()
    for chip in grupo['chip']:
        _, lab = CACHE[chip]
        m_otsu.update(otsu_mask(chip), lab)
        m_red.update(predict(model_final, chip), lab)
    r = m_red.compute()
    filas[evento] = {'chips': len(grupo), 'pct_agua': grupo['pct_agua'].mean(), 'pct_sin_dato': grupo['pct_sin_dato'].mean(),
                     'iou_otsu': m_otsu.compute()['iou'], 'iou_red': r['iou'], 'recall_red': r['recall'], 'comision_red': r['comision']}

por_evento = pd.DataFrame(filas).T.sort_values('iou_red')
por_evento.round(3)

## 19. Los chips donde peor le va (para el análisis de fallos)

Se calcula el IoU de **cada chip** de test y Bolivia y se muestran los 6 peores (entre los que tienen al menos 1 % de agua; en un chip casi sin agua el IoU no dice mucho).

Mapa de errores:
- **Azul:** agua bien detectada (acierto).
- **Rojo:** agua **inventada** (comisión, falso positivo).
- **Naranja:** agua **no vista** (omisión, falso negativo).
- **Beige:** no agua bien detectada. **Gris:** sin dato.

Estos chips sirven para el **punto 5 del examen**: buscar ahí los tipos de fallo (sombra de radar, doble rebote en ciudades, suelo húmedo, vegetación inundada).

In [ ]:
filas = []
for r in chips[chips['split'].isin(['test', 'bolivia'])].itertuples():
    _, lab = CACHE[r.chip]
    m = WaterMetrics()
    m.update(predict(model_final, r.chip), lab)
    filas.append({'chip': r.chip, 'split': r.split, 'pct_agua': r.pct_agua, **m.compute(), 'tp': m.tp, 'fp': m.fp, 'fn': m.fn})

por_chip = pd.DataFrame(filas)
por_chip.to_csv(f'{OUT}/por_chip_test_bolivia.csv', index=False)

# Métrica de los autores de Sen1Floods11: IoU promedio sobre los chips (sin contar chips donde no hay agua ni predicción)
con_algo = por_chip[por_chip[['tp', 'fp', 'fn']].sum(axis=1) > 0]
print('IoU promedio por chip:', con_algo.groupby('split')['iou'].mean().round(4).to_dict())

peores = por_chip[por_chip['pct_agua'] >= 1].nsmallest(6, 'iou')
peores[['chip', 'split', 'pct_agua', 'iou', 'recall', 'comision']].round(3)

In [ ]:
def show_errores(ax, pred, lab, title):
    rgb = np.zeros((*lab.shape, 3))
    rgb[:] = [0.89, 0.84, 0.72]                      # no agua bien detectada (beige)
    rgb[pred & (lab == 1)] = [0.12, 0.44, 0.70]      # acierto de agua (azul)
    rgb[pred & (lab == 0)] = [0.84, 0.15, 0.16]      # agua inventada (rojo)
    rgb[~pred & (lab == 1)] = [1.00, 0.55, 0.00]     # agua no vista (naranja)
    rgb[lab == -1] = [0.60, 0.60, 0.60]              # sin dato (gris)
    ax.imshow(rgb); ax.set_title(title); ax.axis('off')


fig, axes = plt.subplots(len(peores), 4, figsize=(16, 4 * len(peores)))
for row, r in zip(axes, peores.itertuples()):
    img, lab = CACHE[r.chip]
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0);  row[0].set_title(f'{r.chip} · VV'); row[0].axis('off')
    row[1].imshow(img[1], cmap='gray', vmin=-30, vmax=-5); row[1].set_title('VH'); row[1].axis('off')
    show_mask(row[2], lab, 'Real (LabelHand)')
    show_errores(row[3], predict(model_final, r.chip), lab, f'Errores · IoU {r.iou:.2f}')
plt.tight_layout(); plt.show()

## Criterio para cerrar la evaluación en Sen1Floods11

- [ ] Sección 13: ¿E6 supera a E3 por más de 0.02? Anotar la arquitectura.
- [ ] Sección 15: ¿100 épocas superan a 40 por más de 0.02? Anotar las épocas.
- [ ] Sección 17: **IoU de la red frente a Otsu** en test y en Bolivia, y cuánto aportó sumar valid (fila "Solo train" frente a "Red").
- [ ] **Brecha de generalización** = IoU en test − IoU en Bolivia. Si es grande, el modelo se apoya en rasgos propios de los eventos que vio.
- [ ] Sección 18: ¿los peores eventos son los que tienen poca agua o mucho "sin dato"?
- [ ] Sección 19: identificar **al menos 3 tipos de fallo** y anotar un chip de ejemplo para cada uno (punto 5 del examen).

Lo siguiente es **Lima** (Fase 4): aplicar este mismo `model_final` y `normalize` (con `norm_stats.json` de train) a la escena del 17 de marzo de 2017, sin reentrenar.

## Resultados (15-oct) ✅

### Sección 13 (arquitectura) y 15 (épocas), en valid

| | IoU | Precisión | Recall | iou_ult5 | Decisión |
|---|---|---|---|---|---|
| E3 · U-Net, 40 épocas | **0.6425** | 0.787 | **0.778** | 0.618 | ✅ se queda |
| E6 · Attention U-Net | 0.6425 (+0.0000) | 0.810 | 0.757 | 0.618 | empate → se queda E3 (más simple) |
| E7 · U-Net, 100 épocas | 0.6391 (−0.003) | 0.835 | 0.731 | **0.626** | no mejora → 40 épocas |

- Las *attention gates* **no mejoran** el IoU: solo mueven el equilibrio hacia más precisión y menos recall. En los 4 chips de valid, E3 y E6 son casi idénticos.
- Con 100 épocas las curvas se estabilizan (`iou_ult5` más alto), pero el mejor IoU no sube, y la red se vuelve más conservadora. **Con 250 chips, el límite son los datos, no la arquitectura ni el tiempo de entrenamiento.**

### Sección 17: test y Bolivia

| | IoU test | IoU Bolivia | Recall Bolivia | Comisión Bolivia |
|---|---|---|---|---|
| Otsu | 0.555 | **0.718** | 0.870 | 0.197 |
| Solo train (E3) | 0.663 | 0.702 | 0.798 | 0.146 |
| **Red = FINAL (train+valid)** | **0.670** | 0.616 | 0.667 | 0.110 |

- **En test (punto 3)**, la red supera a Otsu por +0.12 de IoU. Sumar valid aporta +0.007, dentro del ruido entre corridas.
- **Matriz de confusión de test:** 563 mil píxeles de agua no vista frente a 397 mil de agua inventada. **La red falla más por omisión que por comisión.** En Bolivia es aún más marcado: 151 mil frente a 37 mil (4 veces más), es decir, la red es conservadora.
- **En Bolivia, Otsu gana.** Ojo: el umbral de Otsu de Sen1Floods11 **se ajustó por evento** (`VH_thresh` de la metadata), así que conoce cada escena. La red, en cambio, nunca vio Bolivia.
- **FINAL cae 0.086 frente a "solo train" en Bolivia** con la misma receta. En test, los dos están a menos de 0.01. Bolivia tiene solo 15 chips, y fuera de distribución **la variación entre corridas es mucho mayor**.
- **IoU promedio por chip** (la métrica de los autores): test 0.400, Bolivia 0.366. Es mucho menor que el IoU por píxel, porque los chips con poca agua sacan IoU bajo y pesan lo mismo que los demás.

### Sección 18: por evento

- **Mejores** (IoU > 0.85): Nigeria, Sri Lanka, Mekong. Tienen agua abierta grande y oscura.
- **Peores:**
  - Pakistan 0.19, con comisión 0.55: inventa agua.
  - Somalia 0.42 y Ghana 0.53, con recall < 0.6: no ven el agua.
  - USA 0.55: hay poca agua (3 %) y en ríos finos.
- **No depende del "sin dato":** Pakistan (30 %) está entre los peores y Nigeria (28 %) es el mejor. **Depende del tipo de agua y de paisaje.**

### Sección 19: tipos de fallo (punto 5 del examen)

| Tipo de fallo | Qué pasa en el radar | Ejemplos |
|---|---|---|
| **1. Agua que no se ve oscura** (vegetación inundada, agua somera o rugosa) | La vegetación sobre el agua produce **doble rebote** y la superficie se ve gris o brillante, no negra → **omisión** | `Ghana_313799`, `Somalia_699062`, `Bolivia_312675` |
| **2. Superficies lisas que no son agua** (campos de cultivo, suelo húmedo o desnudo) | Reflejan el radar como un espejo, igual que el agua → **comisión** | `Pakistan_70625` (manchas rojas), `Paraguay_76868` (valid) |
| **3. Ríos y canales finos** (1–3 píxeles de ancho) | A 10 m por píxel, el río se mezcla con la orilla y la red no lo marca → **omisión** | `USA_670826`, el canal de `Pakistan_70625` |
| Pendiente: **sombra de radar** | Las laderas opuestas al satélite no reciben señal y se ven oscuras | Buscarla en **Lima** (cerros junto al Rímac) |


## 20. ¿Cuánto cambia el resultado si se repite el entrenamiento? (3 semillas)

La **semilla** fija el azar del entrenamiento: los pesos iniciales, el orden de los chips y los recortes y giros de las aumentaciones. Con otra semilla, la misma receta da una red algo distinta.

En la sección 17, dos redes con la misma receta quedaron a menos de 0.01 en test, pero a **0.086 en Bolivia**. Con una sola corrida no sabemos si eso es por sumar valid o simplemente por el azar. Por eso:
- Se repiten **las dos recetas** ("solo train" y "train + valid") con **3 semillas**: 42 (la que ya tenemos), 1 y 2.
- Se reporta **media ± desviación**: es la forma honesta de comparar cuando las diferencias son pequeñas.
- Cada entrenamiento tarda ≈ 3 minutos → son 4 entrenamientos nuevos, ≈ 15 minutos.

Esto **no elige ningún modelo**: solo mide cuánto varía. Para Lima se usarán **las 3 semillas de `FINAL`** y se reportará el promedio.

In [ ]:
SEMILLAS = [42, 1, 2]          # 42 = las corridas que ya están en Drive

def con_semilla(nombre, s):
    return nombre if s == 42 else f'{nombre}_s{s}'

for s in SEMILLAS[1:]:
    run_experiment(con_semilla(ELEGIDO, s), loss_name='bce_dice', epochs=EPOCAS, attention=ATT, seed=s)
    run_experiment(con_semilla(FINAL, s), loss_name='bce_dice', epochs=EPOCAS, attention=ATT,
                   train_splits=('train', 'valid'), seed=s)

In [ ]:
filas = []
for s in SEMILLAS:
    for receta, nombre in [('Solo train', ELEGIDO), ('Train + valid', FINAL)]:
        m = load_model(con_semilla(nombre, s))
        fila = {'receta': receta, 'semilla': s}
        if receta == 'Solo train':          # train + valid ya vio valid: su número de valid no vale
            fila['iou_valid'] = evaluate(m, valid_loader, masked_bce)[1].compute()['iou']
        for split in ['test', 'bolivia']:
            met = eval_split(m, split).compute()
            fila[f'iou_{split}'], fila[f'recall_{split}'] = met['iou'], met['recall']
        filas.append(fila)

semillas = pd.DataFrame(filas)
semillas.to_csv(f'{OUT}/resultados_semillas.csv', index=False)
semillas.round(4)

### Media ± desviación por receta

Regla para leerla: si la diferencia entre las dos recetas es **menor que su desviación**, no hay evidencia de que una sea mejor que la otra.

In [ ]:
resumen = semillas.groupby('receta')[['iou_valid', 'iou_test', 'iou_bolivia', 'recall_test', 'recall_bolivia']].agg(['mean', 'std'])
print('Otsu → IoU test:', round(MET['Otsu · test'].compute()['iou'], 4), '| IoU Bolivia:', round(MET['Otsu · bolivia'].compute()['iou'], 4))
resumen.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, split in zip(axes, ['test', 'bolivia']):
    for j, (receta, grupo) in enumerate(semillas.groupby('receta')):
        ax.scatter([j] * len(grupo), grupo[f'iou_{split}'], s=60, label=receta)
        ax.hlines(grupo[f'iou_{split}'].mean(), j - 0.2, j + 0.2, color='black')
    ax.axhline(MET[f'Otsu · {split}'].compute()['iou'], color='gray', ls='--', label='Otsu')
    ax.set_xticks([0, 1]); ax.set_xticklabels(sorted(semillas['receta'].unique()))
    ax.set_title(f'IoU en {split} (cada punto = una semilla, raya = media)'); ax.grid(alpha=.3)
axes[0].legend()
plt.tight_layout(); plt.show()

### Qué anotar para el informe

- [ ] **IoU en test** de cada receta como media ± desviación (punto 3 del examen).
- [ ] **¿Sumar valid ayuda?** Solo si la diferencia de medias es mayor que las desviaciones.
- [ ] **Desviación en test frente a Bolivia:** si en Bolivia es mucho mayor, el resultado fuera de distribución depende del azar del entrenamiento. Es una limitación que hay que declarar.
- [ ] Para **Lima** se evalúan las 3 semillas de `FINAL`: `[con_semilla(FINAL, s) for s in SEMILLAS]`.